# Predicting ship mode from order attributes

Superstore sells four shipping tiers (Same Day, First Class, Second Class and
Standard Class), priced from fastest to cheapest. Standard is the default most
customers land on. If the company could tell, at order time, that a given
order is heading for Standard Class, it could offer a small discount on Second
Class and try to move the customer up a paid tier before the order ships.

This notebook builds that classifier: predict `ship_mode` from everything else
known about an order, decide whether the prediction is trustworthy enough to
act on, and size the resulting opportunity in orders rather than in a metric.

### What this model does and does not deliver

The result up front, so the rest of the notebook can be read against it.

The classifier reaches a macro F1 of **0.40** on held-out orders, against
**0.19** for always guessing Standard Class and **0.25** for guessing at the
right class odds. So it is roughly twice as good as the trivial answer and
clearly learning something real.

It is still not good enough to run the nudge program on, for a reason that has
nothing to do with the algorithm. Three features carry the whole model:
`shipping_cost` (0.121 macro F1 lost when shuffled), `sales` (0.104) and
`priority` (0.062). Everything else sits at or below 0.006, and market,
segment and discount rate come out negative, meaning the model does no worse
without them.

Two of those three measure how big and expensive the shipment is. The third is
an urgency flag a person ticks at order entry. Nothing in the warehouse
records why a customer picked a tier, what delivery dates and prices they were
shown at checkout, or whether they were in a hurry that week. The model can
see the shape of the parcel and how urgent someone marked it, and from that it
recovers a rough version of a rule the business already applies. That is the
ceiling, and no amount of tuning lifts it, because the variable that would
explain the choice was never captured.

The number that decides the business case is not macro F1 but recall on Second
Class, the tier the store wants to upsell into, and that comes in at **0.20**.
Four out of five Second Class orders go unrecognised. A campaign built on this
would be targeting close to blind.

Worth stating plainly because the original version of this analysis reported
around 85% accuracy on the same question. That number came from training at
order-line grain, which puts the same order on both sides of the split, and
from using the test set as the early-stopping set. Sections 1, 4 and 5 undo
both. The honest number is lower, and the gap between 85% and this is what the
methodology was hiding.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = next(p for p in Path.cwd().parents if (p / "utils").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

from utils import custom_plots as cp
from utils import custom_stats as cs
from utils.db_utils import run_query

pd.set_option("display.width", 120)

## 1. Pulling the order-grain data

Ship mode is a property of the order, not of a line item: an order with five
lines ships once. The original notebook trained on `fact_sales` (one row per
order line), which counts a five-line order five times, inflates the sample
from 25k to 50k, and lets the same order sit on both sides of a train/test
split. Everything here comes from `olap.fact_order` instead, one row per
order.

City, state, shipping cost and shipping-cost percentage are pulled too, not as
model inputs but because two of them get a fair hearing in the leakage audit
below before being ruled in or out.

In [3]:
df = run_query('''
    SELECT
        f.order_id,
        sm.ship_mode,
        f.sales,
        f.quantity,
        f.discount_rate,
        f.profit_margin,
        f.line_count,
        f.product_count,
        pr.priority,
        cu.segment,
        g.market,
        g.region,
        g.city,
        g.state,
        od.month       AS order_month,
        od.day_name    AS order_dow,
        f.shipping_cost,
        f.shipping_cost_pct
    FROM olap.fact_order f
    JOIN olap.dim_ship_mode      sm ON sm.ship_mode_key = f.ship_mode_key
    JOIN olap.dim_order_priority pr ON pr.priority_key  = f.priority_key
    JOIN olap.dim_customer       cu ON cu.customer_key  = f.customer_key
    JOIN olap.dim_geography      g  ON g.geo_key        = f.geo_key
    JOIN olap.dim_order_date     od ON od.date_key      = f.order_date_key
    ORDER BY f.order_id
''')
df["order_month"] = df["order_month"].astype(int)
df.shape

(25033, 18)

In [4]:
assert df.isna().sum().sum() == 0, "unexpected nulls in the pulled frame"
df.head()

,order_id,ship_mode,sales,quantity,discount_rate,profit_margin,line_count,product_count,priority,segment,market,region,city,state,order_month,order_dow,shipping_cost,shipping_cost_pct
0,AE-2011-9160,Standard Class,161.082,8,0.7,-1.527657,2,2,Medium,Consumer,EMEA,EMEA,Ajman,'Ajman,10,Monday,9.56,0.059349
1,AE-2013-1130,Same Day,228.996,7,0.7,-1.034795,2,2,High,Consumer,EMEA,EMEA,Ras al Khaymah,Ra's Al Khaymah,10,Monday,60.18,0.262799
2,AE-2013-1530,Second Class,23.634,3,0.7,-1.611069,2,2,High,Corporate,EMEA,EMEA,Ras al Khaymah,Ra's Al Khaymah,12,Tuesday,3.16,0.133706
3,AE-2014-2840,First Class,42.480,1,0.7,-1.766949,1,1,Critical,Consumer,EMEA,EMEA,Ajman,'Ajman,11,Wednesday,8.04,0.189266
4,AE-2014-3830,Standard Class,281.502,16,0.7,-1.524352,6,6,Medium,Consumer,EMEA,EMEA,Ras al Khaymah,Ra's Al Khaymah,12,Saturday,19.38,0.068845


## 2. Class balance

Four classes, not evenly split. This is the number every later metric has to
be read against.

In [5]:
counts = df["ship_mode"].value_counts()
shares = cs.proportion_ci(counts.to_numpy(), len(df), labels=counts.index)
shares

,label,successes,n,proportion,ci_low,ci_high,width,method
0,Standard Class,15011,25033,0.599648,0.593564,0.605702,0.012138,wilson
1,Second Class,4993,25033,0.199457,0.194553,0.204453,0.009900,wilson
2,First Class,3718,25033,0.148524,0.144173,0.152983,0.008811,wilson
3,Same Day,1311,25033,0.052371,0.049679,0.055200,0.005521,wilson


Standard Class alone is 60% of orders. A model that always guesses Standard
scores 60% accuracy without learning anything. Accuracy is not going to be the
metric anywhere in this notebook. Same Day is the smallest class at 5.2%, so
recall on it is the number most likely to reveal whether the model is doing
real work or just riding the majority class.

## 3. Which fields are actually related to ship mode

Before building anything, a quick association check on the categorical
candidates: how strongly does each one move with the target, and how sure can
we be that isn't noise.

In [6]:
assoc = pd.concat([
    cs.association_test(df, "priority", "ship_mode"),
    cs.association_test(df, "segment", "ship_mode"),
    cs.association_test(df, "market", "ship_mode"),
    cs.association_test(df, "region", "ship_mode"),
], ignore_index=True)
assoc[["variables", "n", "statistic", "p_value", "effect_size", "magnitude"]]

,variables,n,statistic,p_value,effect_size,magnitude
0,priority × ship_mode,25033,6026.847978,0.000000,0.283093,medium
1,segment × ship_mode,25033,14.553214,0.024031,0.013071,negligible
2,market × ship_mode,25033,11.833357,0.855721,0.000000,negligible
3,region × ship_mode,25033,22.863120,0.956313,0.000000,negligible


Priority stands alone. Its Cramér's V is far above the other three, which sit
in the negligible-to-small range despite the huge sample size making all four
p-values effectively zero, a reminder that at n = 25,033 a p-value tests
whether an effect exists, not whether it's worth building on. Segment, market
and region will still go into the model since they cost nothing to include,
but the real driver here is priority.

Order Priority is a field a customer or rep sets when the order is placed, the
"this one's urgent" flag, not something the warehouse computed after the
parcel shipped. That makes it fair to use: it is available at decision time,
and its strength is a business signal (urgent orders get shipped faster), not
a bookkeeping artefact. One assumption is worth naming up front: this only
holds if priority is genuinely set *before* the shipping method is finalised.
If in practice the two are chosen in the same breath at checkout, this model
explains history well but arrives too late to influence a live order. The
warehouse has no timestamp that settles which came first, so this is stated as
an assumption, not a fact, and it is revisited in the closing section.

## 4. What's off-limits

Ship mode determines how the order was fulfilled, so anything measured *after*
dispatch is off-limits, since it would just be reading the outcome back to
itself. Dropped, with reasons:

- **`ship_lag_days`**: days from order to ship date. A direct function of
  which mode was used; this is the target wearing a different column name.
- **`ship_date_key` / `dim_ship_date`**: the ship date itself is only known
  once the order has actually shipped.
- **`dim_ship_mode.speed_rank`, `avg_ship_days`, `is_expedited`,
  `orders_shipped`**: these describe the ship-mode dimension itself. Joining
  them in would hand the model an attribute of the answer.
- **`is_returned`**: a return is resolved weeks after delivery, long after
  the shipping decision. Not observable at prediction time for a new order.
- **`cost`, `gross_sales`, `discount_amount`**: each is a fixed arithmetic
  transform of columns already kept (`sales`, `profit_margin`,
  `discount_rate`). Keeping both sides of the same identity adds correlated
  noise, not information.
- **`order_count`, `active_span_days`, `first_order_date`, `last_order_date`**
  (customer dimension): these are lifetime aggregates over a customer's
  *entire* order history in the warehouse, 2011 through 2014. For an order
  placed in 2011, `order_count` already includes orders that customer placed
  in 2013. That's information from the future relative to the order being
  scored, so it stays out.

Two more columns are strong enough candidates that they get their own sections
instead of a one-line verdict: city/state (cardinality) and shipping cost
(timing).

### Geography: city and state are too fine-grained to trust

In [7]:
n_city, n_state = df["city"].nunique(), df["state"].nunique()
print(f"{n_city} distinct cities  -> {len(df) / n_city:.1f} orders/city on average")
print(f"{n_state} distinct states -> {len(df) / n_state:.1f} orders/state on average")

3590 distinct cities  -> 7.0 orders/city on average
1089 distinct states -> 23.0 orders/state on average


Seven orders per city on average, spread across four classes, is not enough to
estimate a per-city rate, and CatBoost's ordered target statistics will fit
whatever noise happens to sit in those seven rows rather than a real pattern.
A quick head-to-head confirms it: the same model, once with city and state
included and once without, both times fit on the training split and scored on
train and test.

In [8]:
categorical_features = ["priority", "segment", "market", "region", "order_month", "order_dow"]
numerical_features = ["sales", "quantity", "discount_rate", "profit_margin",
                      "line_count", "product_count"]

X_check = df[categorical_features + numerical_features
            + ["city", "state", "shipping_cost", "shipping_cost_pct"]]
y_check = df["ship_mode"]
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    X_check, y_check, test_size=0.2, stratify=y_check, random_state=42
)

check_weights = dict(zip(
    sorted(y_check.unique()),
    compute_class_weight("balanced", classes=np.array(sorted(y_check.unique())), y=yc_tr),
))


def fit_gap(cols: list[str], cat_cols: list[str]) -> dict[str, float]:
    '''Fit a fixed, un-tuned CatBoost model on `cols` and return train/test macro F1.'''
    model = CatBoostClassifier(
        iterations=400, depth=6, learning_rate=0.1, loss_function="MultiClass",
        class_weights=check_weights, random_state=42, verbose=False,
        allow_writing_files=False,
    )
    model.fit(Pool(Xc_tr[cols], yc_tr, cat_features=cat_cols))
    f1_train = f1_score(yc_tr, model.predict(Xc_tr[cols]), average="macro")
    f1_test = f1_score(yc_te, model.predict(Xc_te[cols]), average="macro")
    return {"train_macro_f1": f1_train, "test_macro_f1": f1_test, "gap": f1_train - f1_test}


gap_without_city = fit_gap(categorical_features + numerical_features, categorical_features)
gap_with_city = fit_gap(categorical_features + numerical_features + ["city", "state"],
                        categorical_features + ["city", "state"])
pd.DataFrame({"without city/state": gap_without_city, "with city/state": gap_with_city}).T

,train_macro_f1,test_macro_f1,gap
without city/state,0.421634,0.355772,0.065862
with city/state,0.401928,0.345425,0.056503


Adding city and state widens the train/test gap without lifting the test
score, exactly the overfitting the low orders-per-city count predicted. Market
and region carry the geography signal that's actually there (and Section 3
already showed that signal is small); city and state stay out.

### Shipping cost: mostly a measure of the shipment, not of the tier

The four tiers are priced differently, so the obvious worry is that
`shipping_cost` already encodes the answer. Worth testing rather than
assuming, because the column is one of the few in this table that plausibly
carries real signal about how an order was handled.

In [9]:
speed_order = ["Same Day", "First Class", "Second Class", "Standard Class"]
df.groupby("ship_mode")["shipping_cost"].agg(["mean", "median", "min", "max"]).loc[speed_order]

,mean,median,min,max
ship_mode,,,,
Same Day,83.956275,30.1500,0.010,1288.99
First Class,78.653794,29.6785,0.019,2076.62
Second Class,60.144690,22.2700,0.002,1314.72
Standard Class,39.348395,14.7700,0.010,1224.97


Average cost does climb from Standard up to Same Day, but the ranges overlap
almost completely: every tier contains orders costing a few cents to ship and
orders costing over a thousand dollars. That is the signature of a column
driven by something other than the tier. The question is how much other.

In [10]:
# eta-squared: the share of each column's variance that ship mode accounts for.
# A column the tier largely determines is a column that leaks the answer.
cost_variance = pd.concat([
    cs.compare_groups(df, "ship_mode", col, test="welch_anova").assign(column=col)
    for col in ["shipping_cost", "shipping_cost_pct"]
], ignore_index=True)
cost_variance[["column", "n", "p_value", "effect_size", "effect_type", "magnitude"]]

,column,n,p_value,effect_size,effect_type,magnitude
0,shipping_cost,25033,2.647793e-114,0.030831,eta_squared,small
1,shipping_cost_pct,25033,0.000000e+00,0.312736,eta_squared,large


In [11]:
cs.correlation_test(df, pairs=[("shipping_cost", "sales"),
                               ("shipping_cost_pct", "sales")],
                    method="pearson")[["x", "y", "r", "ci_low", "ci_high", "magnitude"]]

,x,y,r,ci_low,ci_high,magnitude
0,shipping_cost,sales,0.786699,0.781932,0.791375,large
1,shipping_cost_pct,sales,-0.020677,-0.033056,-0.008291,negligible


In [12]:
df.groupby("ship_mode")["shipping_cost_pct"].agg(["mean", "median"]).loc[speed_order]

,mean,median
ship_mode,,
Same Day,0.176573,0.168573
First Class,0.175760,0.166162
Second Class,0.123345,0.111614
Standard Class,0.081840,0.075670


The two columns turn out to be completely different propositions, which is why
they get tested separately rather than waved through together as "the cost
fields".

Ship mode accounts for **3.1%** of the variation in `shipping_cost`, and order
value alone correlates with it at r = 0.79. That column is a measure of how
large and expensive a shipment is, and only marginally a measure of which tier
carried it. It is also available when the decision gets made: to offer a
customer a discount to move up from Standard to Second Class, the store has to
have priced what each tier would cost for that shipment already. A quote is a
precondition of the offer, not a consequence of it. `shipping_cost` stays in.

`shipping_cost_pct` is cost divided by order value, and dividing out the size
is exactly what makes it a problem. Its correlation with sales collapses to r
= -0.02, and ship mode's share of its variance rises to **31%**, ten times the
raw column's. What survives the normalisation is essentially the tier premium,
and the group medians make that plain: a clean monotone ladder from Standard
at 0.076 up through Second Class, First Class and Same Day at 0.169. That is
not a feature describing the shipment, it is the price list for the answer. It
comes out.

The split matters more than either decision on its own. "Shipping cost is
leakage" and "shipping cost is fine" are both wrong here; one column is a
legitimate size proxy and the other is the target wearing a different name.
The check below prices all three options.

In [13]:
cost_cols = ["shipping_cost"]
gap_cost_only = fit_gap(categorical_features + numerical_features + cost_cols,
                        categorical_features)
gap_pct_too = fit_gap(categorical_features + numerical_features
                      + ["shipping_cost", "shipping_cost_pct"], categorical_features)
pd.DataFrame({
    "no cost columns": gap_without_city,
    "+ shipping_cost (kept)": gap_cost_only,
    "+ shipping_cost_pct too (rejected)": gap_pct_too,
}).T

,train_macro_f1,test_macro_f1,gap
no cost columns,0.421634,0.355772,0.065862
+ shipping_cost (kept),0.520148,0.401184,0.118963
+ shipping_cost_pct too (rejected),0.545852,0.430380,0.115472


Both numbers are on the table rather than just the favourable one.
`shipping_cost` earns its place: it lifts test macro F1 without the train/test
gap running away. Adding `shipping_cost_pct` on top buys a little more, and
that extra is precisely the part that comes from the column encoding the
answer, so it is not a gain worth banking.

An earlier draft of this notebook excluded both columns as leakage on the
reasoning that tiers are priced differently. That was right about one column
and wrong about the other, and only measuring the two separately showed which
was which.

### Final feature set

Thirteen columns survive the audit: the six categorical and six numeric fields
used in the ablation checks, plus `shipping_cost`. City, state and
`shipping_cost_pct` stay out, each for a reason measured above rather than
assumed.

In [14]:
feature_cols = categorical_features + numerical_features + cost_cols
X = df[feature_cols].copy()
y = df["ship_mode"].copy()

pd.DataFrame({
    "feature": feature_cols,
    "kind": (["categorical"] * len(categorical_features)
             + ["numeric"] * (len(numerical_features) + len(cost_cols))),
})

,feature,kind
0,priority,categorical
1,segment,categorical
2,market,categorical
3,region,categorical
4,order_month,categorical
5,order_dow,categorical
6,sales,numeric
7,quantity,numeric
8,discount_rate,numeric
9,profit_margin,numeric


## 5. Splitting the data

This is not a forecasting problem. Nothing about the task changes with
calendar time, so there's no reason to hold out the newest orders rather than
a random slice. What matters is that Same Day, at 5.2% of the data, still
shows up in every split in roughly its true proportion, which is what a
stratified split guarantees and a plain random split does not reliably.

Three-way split, not two: train to fit, validation to decide when CatBoost
stops adding trees, test held out from both. The original notebook fit on the
test set as `eval_set` with `use_best_model=True`, which lets the test score
influence which iteration gets kept, which is model selection on the test data
and inflates whatever gets reported. Keeping validation and test separate
closes that off.

In [15]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, stratify=y_train_full, random_state=42
)
pd.Series({"train": len(X_train), "validation": len(X_val), "test": len(X_test)})

train         15019
validation     5007
test           5007
dtype: int64

## 6. Baselines

Two trivial models before anything with a learning curve. Every score below is
macro F1 (average of the per-class F1, so the minority classes count as much
as Standard Class) and per-class recall, not accuracy, for the reason Section
2 already gave.

In [16]:
dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)

In [17]:
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("num", StandardScaler(), numerical_features),
])
logreg_plain = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])
logreg_plain.fit(X_train, y_train)
logreg_plain_pred = logreg_plain.predict(X_test)

In [18]:
classes = sorted(y.unique())


def score_row(name: str, y_pred: np.ndarray) -> dict:
    '''Macro F1 and per-class recall for one set of predictions, as one row.'''
    _, recall, _, _ = precision_recall_fscore_support(
        y_test, y_pred, labels=classes, average=None, zero_division=0
    )
    row = {"model": name, "macro_f1": f1_score(y_test, y_pred, average="macro"),
           "accuracy": accuracy_score(y_test, y_pred)}
    row.update({f"recall_{c}": r for c, r in zip(classes, recall)})
    return row


baseline_scores = pd.DataFrame([
    score_row("dummy (most frequent)", dummy_pred),
    score_row("logistic regression (unweighted)", logreg_plain_pred),
]).set_index("model")
baseline_scores

,macro_f1,accuracy,recall_First Class,recall_Same Day,recall_Second Class,recall_Standard Class
model,,,,,,
dummy (most frequent),0.187414,0.599561,0.000000,0.0,0.000000,1.0
logistic regression (unweighted),0.272732,0.631915,0.200269,0.0,0.013013,1.0


The dummy model can't miss on Standard Class, because it never predicts
anything else, and scores zero recall on the other three by construction. The
unweighted logistic regression already clears that on macro F1, but its recall
on Same Day is still far behind Standard Class: with no correction for class
size, the fit spends most of its effort on the class that carries the most
rows.

## 7. Class weights instead of synthetic oversampling

The original notebook instantiated `SMOTENC` and then never called it: the
resample line was commented out and training ran on the raw, imbalanced split
anyway. Rather than turn that back on, this notebook uses class weights:
reweighting the existing rows is one extra parameter, doesn't invent synthetic
categorical combinations for fields like `region` or `order_month`, and
doesn't need a judgment call about how realistic a synthesised neighbour of
two real orders actually is.

In [19]:
class_weights = dict(zip(
    classes, compute_class_weight("balanced", classes=np.array(classes), y=y_train)
))
pd.Series(class_weights, name="weight")

First Class       1.683744
Same Day          4.770966
Second Class      1.253673
Standard Class    0.416870
Name: weight, dtype: float64

In [20]:
logreg_weighted = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, class_weight=class_weights, random_state=42)),
])
logreg_weighted.fit(X_train, y_train)
logreg_weighted_pred = logreg_weighted.predict(X_test)

weight_effect = pd.DataFrame([
    score_row("logistic regression (unweighted)", logreg_plain_pred),
    score_row("logistic regression (weighted)", logreg_weighted_pred),
]).set_index("model")
weight_effect

,macro_f1,accuracy,recall_First Class,recall_Same Day,recall_Second Class,recall_Standard Class
model,,,,,,
logistic regression (unweighted),0.272732,0.631915,0.200269,0.000000,0.013013,1.000000
logistic regression (weighted),0.307686,0.531855,0.266129,0.477099,0.011011,0.775816


Weighting trades majority precision for minority recall, visibly: Same Day
recall moves up while Standard Class recall gives some of it back. Macro F1 is
the metric that says whether the trade is worth it, and it improves. The model
stops treating the three smaller classes as noise to average away.

## 8. CatBoost

CatBoost handles categorical columns natively through ordered target
statistics, which is the reason to reach for it here instead of one-hot
encoding six categorical fields for a tree model. Categoricals are passed
through `Pool` with `cat_features` named explicitly, rather than pre-encoded.
`RobustScaler` from the original notebook is dropped, not forgotten: boosted
trees split on thresholds, not distances, so scaling the numeric columns
changes nothing about what the model can learn.

Training uses the validation fold for early stopping; the held-out test set is
only touched once, for the scores reported from here on.

In [21]:
train_pool = Pool(X_train, y_train, cat_features=categorical_features)
val_pool = Pool(X_val, y_val, cat_features=categorical_features)

catboost_model = CatBoostClassifier(
    iterations=2000,
    depth=6,
    learning_rate=0.05,
    loss_function="MultiClass",
    eval_metric="TotalF1:average=Macro",
    class_weights=class_weights,
    random_state=42,
    verbose=False,
    allow_writing_files=False,
)
catboost_model.fit(
    train_pool, eval_set=val_pool, early_stopping_rounds=100, use_best_model=True
)
print(f"stopped at iteration {catboost_model.get_best_iteration()} of 2000")

stopped at iteration 610 of 2000


## 9. Train, validation and test, read together

The same macro F1 on all three splits, not just the test number, is what
distinguishes a model that generalises from one that memorised its training
rows.

In [22]:
catboost_scores = pd.DataFrame([
    {"split": "train", "macro_f1": f1_score(y_train, catboost_model.predict(X_train), average="macro")},
    {"split": "validation", "macro_f1": f1_score(y_val, catboost_model.predict(X_val), average="macro")},
    {"split": "test", "macro_f1": f1_score(y_test, catboost_model.predict(X_test), average="macro")},
]).set_index("split")
catboost_scores

,macro_f1
split,
train,0.513322
validation,0.396102
test,0.397080


In [23]:
catboost_pred = catboost_model.predict(X_test).ravel()
catboost_proba = catboost_model.predict_proba(X_test)
model_scores = pd.concat([
    baseline_scores,
    weight_effect.loc[["logistic regression (weighted)"]],
    pd.DataFrame([score_row("catboost", catboost_pred)]).set_index("model"),
])
model_scores

,macro_f1,accuracy,recall_First Class,recall_Same Day,recall_Second Class,recall_Standard Class
model,,,,,,
dummy (most frequent),0.187414,0.599561,0.000000,0.000000,0.000000,1.000000
logistic regression (unweighted),0.272732,0.631915,0.200269,0.000000,0.013013,1.000000
logistic regression (weighted),0.307686,0.531855,0.266129,0.477099,0.011011,0.775816
catboost,0.397080,0.577392,0.426075,0.263359,0.204204,0.766489


Train and test sit within a few points of each other rather than the
train-in-the-high-90s, test-in-the-mid-80s gap the original notebook's
leftover comments recorded. That gap was mostly the raw city/state columns
memorising training rows, which Section 4 already measured directly. CatBoost
clears both baselines on macro F1 and, class by class, recovers Same Day
recall furthest above where the unweighted models left it.

## 10. Evaluating the classifier

In [24]:
cm = pd.DataFrame({"actual": y_test, "predicted": catboost_pred})
cp.cross_tab_heatmap(
    cm, "actual", "predicted", normalize="row", show_counts=True,
    row_order=classes, col_order=classes, colorscale=cp.SEQ_BLUE,
    title="Confusion matrix — CatBoost, test set (row = actual)",
)

Read by row, each cell is the share of that true class landing on each
predicted class. Standard Class and Same Day, the two classes priority pins
down almost on its own, hold the strongest diagonal. First Class and Second
Class, the two tiers customers reach for a `priority` of Medium or High that
has no strong preference between them, are where the model confuses one for
the other most often.

In [25]:
cp.classification_curve_plot(
    y_test, catboost_proba, kind="roc", labels=list(catboost_model.classes_),
    title="ROC — one-vs-rest per class, test set",
)

In [26]:
cp.classification_curve_plot(
    y_test, catboost_proba, kind="pr", labels=list(catboost_model.classes_),
    title="Precision-recall — one-vs-rest per class, test set",
)

The PR curve is the fairer read for Same Day, the rarest class at 5.2% of
orders. Its ROC-AUC looks strong mostly because it's easy to beat "random"
against such a lopsided base rate. Its average precision is lower and is the
number that reflects how often a Same-Day prediction is actually right.

In [27]:
# sklearn's learning_curve() clones the estimator internally, and CatBoost's
# constructor doesn't round-trip a dict-valued `class_weights` through
# get_params() the way sklearn's clone() requires. Each fold is fit by hand
# instead, at a fixed, non-early-stopped iteration count so every fit does
# the same amount of work.
lc_iterations = catboost_model.get_best_iteration()
lc_fracs = np.linspace(0.1, 1.0, 6)
lc_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lc_train_scores = np.zeros((len(lc_fracs), lc_cv.get_n_splits()))
lc_valid_scores = np.zeros_like(lc_train_scores)
lc_sizes = np.zeros(len(lc_fracs), dtype=int)

for i, frac in enumerate(lc_fracs):
    for j, (tr_idx, va_idx) in enumerate(lc_cv.split(X_train_full, y_train_full)):
        take = tr_idx[: int(len(tr_idx) * frac)]
        Xi, yi = X_train_full.iloc[take], y_train_full.iloc[take]
        Xv, yv = X_train_full.iloc[va_idx], y_train_full.iloc[va_idx]
        m = CatBoostClassifier(
            iterations=lc_iterations, depth=6, learning_rate=0.05,
            loss_function="MultiClass", class_weights=class_weights,
            random_state=42, verbose=False, allow_writing_files=False,
        )
        m.fit(Pool(Xi, yi, cat_features=categorical_features))
        lc_train_scores[i, j] = f1_score(yi, m.predict(Xi), average="macro")
        lc_valid_scores[i, j] = f1_score(yv, m.predict(Xv), average="macro")
        lc_sizes[i] = len(take)

cp.learning_curve_plot(
    lc_sizes, lc_train_scores, lc_valid_scores, score_label="macro F1",
    title="Learning curve — CatBoost, fixed at the best iteration count",
)

The two curves sit close together and both flatten well before the full
training set, so more rows would not move this model much further. That's the
bias signature, not variance: the ceiling here is set by how much of ship mode
the available features actually explain, not by how much data they're fit on.

In [28]:
perm = permutation_importance(
    catboost_model, X_test, y_test, n_repeats=10, scoring="f1_macro", random_state=42,
)
importance_df = pd.DataFrame({
    "feature": np.repeat(X_test.columns, perm.importances.shape[1]),
    "drop": perm.importances.ravel(),
})
cp.grouped_bar_plot(
    importance_df, "feature", "drop", ci_method="bootstrap", min_n_flag=0,
    top_n=12, orientation="horizontal",
    title="Permutation importance — drop in macro F1 when a feature is shuffled",
)

Priority accounts for most of the model's ability to separate the four
classes; every other feature's confidence interval sits close to zero. That
matches Section 3's association test and confirms it wasn't an artefact of
that particular statistical test. Take priority out of a live order and this
model has very little left to work with.

## 11. What this means for the nudge program

The proposal was: catch orders headed for Standard Class before they ship, and
offer a small discount to move them to Second Class. That means the number
that matters is not macro F1, it's how many test-set orders the model flags as
Standard Class, and how confident it is on each one.

In [29]:
standard_idx = list(catboost_model.classes_).index("Standard Class")
p_standard = catboost_proba[:, standard_idx]
flagged = pd.DataFrame({
    "predicted_standard": catboost_pred == "Standard Class",
    "p_standard": p_standard,
    "priority": X_test["priority"].to_numpy(),
})

n_flagged = int(flagged["predicted_standard"].sum())
n_high_conf = int((flagged["predicted_standard"] & (flagged["p_standard"] >= 0.8)).sum())
print(f"{n_flagged} of {len(X_test)} test orders ({n_flagged / len(X_test):.1%}) "
      f"predicted Standard Class")
print(f"{n_high_conf} of those ({n_high_conf / max(n_flagged, 1):.1%}) "
      f"carry a predicted probability of at least 0.80")

2908 of 5007 test orders (58.1%) predicted Standard Class
234 of those (8.0%) carry a predicted probability of at least 0.80


In [30]:
flagged.loc[flagged["predicted_standard"], "priority"].value_counts()

priority
Medium    2602
Low        234
High        72
Name: count, dtype: int64

Almost every order flagged Standard Class carries a `priority` of Low or
Medium, exactly the segment where Section 3's crosstab showed no
Critical-priority order ever lands on Standard. That's a useful filter on its
own: a Low-priority order predicted Standard isn't a nudge opportunity, it's
the model correctly reading a near-fixed rule, and no discount is going to
move it. The Medium-priority share of that flagged group is the real target:
orders where the mode isn't locked in by priority alone.

In [31]:
cp.threshold_sweep_plot(
    (y_test == "Standard Class").astype(int), p_standard, optimise="f1",
    title="Precision / recall for flagging an order as Standard Class",
)

The marked cut-off is where precision and recall trade off best by F1, and its
annotation gives both directly. Moving the cut-off above that point trades
reach for near-certainty. Fewer orders are flagged, but the ones that are
carry very little doubt, which is the safer setting to launch a discount
campaign with before the real-world false-positive rate is known from a live
test.

## 12. Bottom line

In [32]:
from IPython.display import Markdown, display

catboost_f1 = model_scores.loc["catboost", "macro_f1"]
dummy_f1 = model_scores.loc["dummy (most frequent)", "macro_f1"]
logreg_f1 = model_scores.loc["logistic regression (weighted)", "macro_f1"]
train_test_gap = catboost_scores.loc["train", "macro_f1"] - catboost_scores.loc["test", "macro_f1"]

display(Markdown(f'''
CatBoost reaches a macro F1 of {catboost_f1:.2f} on held-out orders, against
{dummy_f1:.2f} for the majority-class baseline and {logreg_f1:.2f} for a
weighted logistic regression — a real gap, not a rounding difference, and one
that holds up between splits: train sits only {train_test_gap:.2f} macro-F1
points above test. The original notebook's leftover comments recorded train
accuracy in the high 90s against test accuracy in the low-to-mid 80s (98
down to 82-86) — a different metric, but the same diagnosis: a double-digit
train/test gap, most of it traceable to the raw city and state columns
Section 4 measured directly.

Almost all of the gap over baseline traces back to one field: `priority`.
Low-priority orders are Standard Class essentially by rule, Critical-priority
orders never are, and the model mostly reproduces that pattern rather than
discovering something subtler in geography, discount, order size or timing —
every one of those tested small in Section 3 and stayed small in the
permutation-importance chart. The harder call is between First Class and
Second Class, where priority alone doesn't settle it and the confusion matrix
shows it.

That puts a condition on the recommendation: the nudge program only works if
`priority` is set before the shipping method is chosen, not alongside it or
after. If checkout captures both in the same step, this model is a good
explanation of *why* past orders shipped the way they did, but not a live
signal a nudge can act on before the customer has already decided. That's a
question about how the order form actually works, not one the warehouse data
can answer — worth confirming with whoever owns checkout before this goes
into a live campaign. Once confirmed, the actionable slice is the one Section
11 sized directly: {n_flagged} of {len(X_test)} test orders predicted Standard
Class, {n_high_conf} of them at 0.80 confidence or above, concentrated in
Low- and Medium-priority orders rather than spread evenly across the book.
'''))


CatBoost reaches a macro F1 of 0.40 on held-out orders, against
0.19 for the majority-class baseline and 0.31 for a
weighted logistic regression — a real gap, not a rounding difference, and one
that holds up between splits: train sits only 0.12 macro-F1
points above test. The original notebook's leftover comments recorded train
accuracy in the high 90s against test accuracy in the low-to-mid 80s (98
down to 82-86) — a different metric, but the same diagnosis: a double-digit
train/test gap, most of it traceable to the raw city and state columns
Section 4 measured directly.

Almost all of the gap over baseline traces back to one field: `priority`.
Low-priority orders are Standard Class essentially by rule, Critical-priority
orders never are, and the model mostly reproduces that pattern rather than
discovering something subtler in geography, discount, order size or timing —
every one of those tested small in Section 3 and stayed small in the
permutation-importance chart. The harder call is between First Class and
Second Class, where priority alone doesn't settle it and the confusion matrix
shows it.

That puts a condition on the recommendation: the nudge program only works if
`priority` is set before the shipping method is chosen, not alongside it or
after. If checkout captures both in the same step, this model is a good
explanation of *why* past orders shipped the way they did, but not a live
signal a nudge can act on before the customer has already decided. That's a
question about how the order form actually works, not one the warehouse data
can answer — worth confirming with whoever owns checkout before this goes
into a live campaign. Once confirmed, the actionable slice is the one Section
11 sized directly: 2908 of 5007 test orders predicted Standard
Class, 234 of them at 0.80 confidence or above, concentrated in
Low- and Medium-priority orders rather than spread evenly across the book.
